# Online Bookstore — SQL Solutions & Findings

This notebook covers the 11 core and 9 advanced questions with runnable PostgreSQL queries, result tables and findings derived from the connected database.

**Prerequisites:** PostgreSQL with populated `books`, `customers` and `orders` tables. Set `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD`, `PGDATABASE`. The notebook does not embed local Windows import paths or credentials.

## 1. Connect and profile data

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
required=["PGHOST","PGUSER","PGPASSWORD","PGDATABASE"]
missing=[k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError("Set PostgreSQL variables: "+", ".join(missing))
engine=create_engine(
    f"postgresql+psycopg2://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}"
    f"@{os.environ['PGHOST']}:{os.getenv('PGPORT','5432')}/{os.environ['PGDATABASE']}"
)
def run_query(sql):
    return pd.read_sql_query(text(sql),engine)
profile=run_query("""
SELECT
 (SELECT COUNT(*) FROM books) AS book_rows,
 (SELECT COUNT(*) FROM customers) AS customer_rows,
 (SELECT COUNT(*) FROM orders) AS order_rows,
 (SELECT COUNT(*) FROM orders o LEFT JOIN books b ON b.book_id=o.book_id WHERE b.book_id IS NULL) AS orders_without_book,
 (SELECT COUNT(*) FROM orders o LEFT JOIN customers c ON c.customer_id=o.customer_id WHERE c.customer_id IS NULL) AS orders_without_customer
""")
display(profile)

## 2. Core Q1 — Books in Fiction genre

In [ ]:
q1=run_query("SELECT book_id,title,author,price,stock FROM books WHERE genre='Fiction' ORDER BY title")
display(q1)
print(f"Finding: {len(q1):,} Fiction book records match the filter.")

## 3. Core Q2 — Books published after 1950

In [ ]:
q2=run_query("SELECT book_id,title,author,published_year FROM books WHERE published_year>1950 ORDER BY published_year DESC")
display(q2.head(30))
print(f"Finding: {len(q2):,} book records have a publication year after 1950.")

## 4. Core Q3 — Customers in Canada

In [ ]:
q3=run_query("SELECT customer_id,name,city,country FROM customers WHERE country ILIKE 'Canada' ORDER BY city,name")
display(q3)
print(f"Finding: {len(q3):,} customers are listed with country Canada.")

## 5. Core Q4 — Orders placed in November 2023

In [ ]:
q4=run_query("""
SELECT order_id,customer_id,book_id,order_date,quantity,total_amount
FROM orders
WHERE order_date >= DATE '2023-11-01' AND order_date < DATE '2023-12-01'
ORDER BY order_date
""")
display(q4)
print(f"Finding: {len(q4):,} order rows were placed in November 2023; recorded order value totals {q4['total_amount'].sum():,.2f}.")

## 6. Core Q5 — Total book stock

In [ ]:
q5=run_query("SELECT SUM(stock) AS total_stock_units, COUNT(*) AS book_skus FROM books")
display(q5)
print("Finding: this is the sum of the stock snapshot in books; confirm whether stock is current stock or pre-order stock before subtracting historical orders.")

## 7. Core Q6 — Most expensive book(s), retaining ties

In [ ]:
q6=run_query("""
SELECT book_id,title,author,price
FROM books
WHERE price=(SELECT MAX(price) FROM books)
ORDER BY title
""")
display(q6)
print(f"Finding: {len(q6):,} book(s) tie at the maximum listed price.")

## 8. Core Q7 — Order lines with quantity above one

In [ ]:
q7=run_query("""
SELECT order_id,customer_id,book_id,order_date,quantity,total_amount
FROM orders WHERE quantity>1 ORDER BY quantity DESC,order_date
""")
display(q7.head(30))
print(f"Finding: {len(q7):,} order rows contain quantity greater than one.")

## 9. Core Q8 — Orders over $20

In [ ]:
q8=run_query("SELECT order_id,order_date,customer_id,total_amount FROM orders WHERE total_amount>20 ORDER BY total_amount DESC")
display(q8.head(30))
print(f"Finding: {len(q8):,} orders exceed 20 in recorded order value.")

## 10. Core Q9 — Available genres

In [ ]:
q9=run_query("SELECT DISTINCT genre FROM books WHERE genre IS NOT NULL ORDER BY genre")
display(q9)
print(f"Finding: {len(q9):,} distinct non-null genre labels are present.")

## 11. Core Q10 — Lowest-stock book(s), retaining ties

In [ ]:
q10=run_query("""
SELECT book_id,title,author,stock
FROM books
WHERE stock=(SELECT MIN(stock) FROM books)
ORDER BY title
""")
display(q10)
print(f"Finding: {len(q10):,} book(s) tie at the lowest stock level.")

## 12. Core Q11 — Total recorded order value

In [ ]:
q11=run_query("SELECT SUM(total_amount) AS total_recorded_order_value, COUNT(*) AS order_rows FROM orders")
display(q11)
print("Finding: total_amount is treated as order-row value. Confirm whether it is line value or whole-order value before calling it revenue.")

## 13. Advanced Q1 — Units sold by genre

In [ ]:
a1=run_query("""
SELECT b.genre,SUM(o.quantity) AS units_sold,
       SUM(o.total_amount) AS recorded_order_value
FROM orders o JOIN books b ON b.book_id=o.book_id
GROUP BY b.genre ORDER BY units_sold DESC
""")
display(a1)
if not a1.empty:
    r=a1.iloc[0]
    print(f"Finding: {r['genre']} leads by recorded units sold ({int(r['units_sold']):,}).")

## 14. Advanced Q2 — Average Fantasy book price

In [ ]:
a2=run_query("SELECT ROUND(AVG(price),2) AS average_fantasy_price,COUNT(*) AS fantasy_skus FROM books WHERE genre='Fantasy'")
display(a2)
print("Finding: average price describes listed Fantasy SKUs, not average customer spend.")

## 15. Advanced Q3 — Customers with at least two order rows

In [ ]:
a3=run_query("""
SELECT c.customer_id,c.name,COUNT(o.order_id) AS order_count,
       SUM(o.total_amount) AS recorded_spend
FROM orders o JOIN customers c ON c.customer_id=o.customer_id
GROUP BY c.customer_id,c.name
HAVING COUNT(o.order_id)>=2
ORDER BY order_count DESC,recorded_spend DESC
""")
display(a3.head(30))
print(f"Finding: {len(a3):,} customers have at least two order rows.")

## 16. Advanced Q4 — Most frequently ordered books

Count both order rows and quantity so “frequent” and “units sold” are not confused.

In [ ]:
a4=run_query("""
SELECT b.book_id,b.title,b.author,
       COUNT(o.order_id) AS order_rows,
       SUM(o.quantity) AS units_ordered
FROM orders o JOIN books b ON b.book_id=o.book_id
GROUP BY b.book_id,b.title,b.author
ORDER BY order_rows DESC,units_ordered DESC
LIMIT 10
""")
display(a4)
if not a4.empty:
    r=a4.iloc[0]
    print(f"Finding: {r['title']} leads by order-row count ({int(r['order_rows']):,}); units ordered provide a separate demand measure.")

## 17. Advanced Q5 — Top three Fantasy books by price

In [ ]:
a5=run_query("SELECT book_id,title,author,price FROM books WHERE genre='Fantasy' ORDER BY price DESC,title LIMIT 3")
display(a5)
print("Finding: these are the three highest-priced Fantasy book records; high price does not automatically mean high demand.")

## 18. Advanced Q6 — Units sold by author

In [ ]:
a6=run_query("""
SELECT b.author,SUM(o.quantity) AS units_sold,
       COUNT(DISTINCT o.order_id) AS order_rows,
       SUM(o.total_amount) AS recorded_order_value
FROM orders o JOIN books b ON b.book_id=o.book_id
GROUP BY b.author ORDER BY units_sold DESC
""")
display(a6.head(20))
if not a6.empty:
    r=a6.iloc[0]
    print(f"Finding: {r['author']} leads by recorded units sold ({int(r['units_sold']):,}).")

## 19. Advanced Q7 — Cities of customers whose cumulative spend exceeds $30

The original question wording refers to customers who spent over $30; this solution sums all recorded order values per customer before selecting cities.

In [ ]:
a7=run_query("""
WITH customer_spend AS (
 SELECT customer_id,SUM(total_amount) AS total_spend
 FROM orders GROUP BY customer_id
)
SELECT DISTINCT c.city,cs.customer_id,cs.total_spend
FROM customer_spend cs JOIN customers c ON c.customer_id=cs.customer_id
WHERE cs.total_spend>30
ORDER BY cs.total_spend DESC,c.city
""")
display(a7)
print(f"Finding: {a7['city'].nunique() if not a7.empty else 0:,} distinct cities contain customers with cumulative recorded spend above 30.")

## 20. Advanced Q8 — Customer with the highest recorded spend

In [ ]:
a8=run_query("""
WITH customer_spend AS (
 SELECT c.customer_id,c.name,SUM(o.total_amount) AS total_spend,
        COUNT(o.order_id) AS order_rows
 FROM customers c JOIN orders o ON o.customer_id=c.customer_id
 GROUP BY c.customer_id,c.name
)
SELECT * FROM customer_spend
WHERE total_spend=(SELECT MAX(total_spend) FROM customer_spend)
ORDER BY customer_id
""")
display(a8)
print("Finding: ties are retained for the highest customer spend.")

## 21. Advanced Q9 — Stock remaining after orders (assumption-sensitive)

In [ ]:
a9=run_query("""
SELECT b.book_id,b.title,b.stock AS stock_snapshot,
       COALESCE(SUM(o.quantity),0) AS recorded_order_units,
       b.stock-COALESCE(SUM(o.quantity),0) AS hypothetical_remaining_stock
FROM books b LEFT JOIN orders o ON o.book_id=b.book_id
GROUP BY b.book_id,b.title,b.stock
ORDER BY b.book_id
""")
display(a9.head(30))
print("Interpretation warning: this subtraction is valid only if books.stock is a pre-order stock baseline and the orders table contains all fulfilled orders in the same period. If stock is already current, subtracting historical orders double-counts depletion.")

## Executive summary
Use the results to compare genre demand, customer spend, repeat order activity, stock levels and high-value books. Confirm order-value semantics, source-data coverage and whether stock is a current snapshot or historical baseline before translating results into revenue or replenishment recommendations.